# Etapa N°0

## 1. Carga y unificación de fuentes bibliográficas

Se importó un archivo Excel con múltiples hojas, cada una correspondiente a diferentes bases de datos y consultas bibliográficas.  
Todas las hojas fueron cargadas simultáneamente y unificadas en una única tabla, preservando la trazabilidad mediante una columna que identifica la hoja de origen.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
import os

path = "/content/drive/MyDrive/Título"
os.listdir(path)


['Bibliografia Inicial.gsheet',
 'Bibliografia Inicial.xlsx',
 'bibliografia_etapa1.csv']

In [ ]:
import pandas as pd

file_path = "/content/drive/MyDrive/Título/Bibliografia Inicial.xlsx"
df = pd.read_excel(file_path)

df.head()

,Item Title,Publication Title,Book Series Title,Journal Volume,Journal Issue,Item DOI,Authors,Publication Year,URL,Content Type
0,AI-Driven Precision Oncology: Integrating Deep...,"Signal, Image and Video Processing",NaN,NaN,NaN,10.1007/s11760-025-04244-y,Faridoddin ShariatyVitalii PavlovMaksim Baranov,2025,https://link.springer.com/article/10.1007/s117...,Article
1,Breast Cancer Histology Image Repositories in ...,Data Science and Applications,NaN,NaN,NaN,10.1007/978-3-032-10940-8_18,Vinita ShahMiral Patel,2026,https://link.springer.com/chapter/10.1007/978-...,Conference paper
2,A federated learning system for precision onco...,Nature Medicine,NaN,NaN,NaN,10.1038/s41591-023-02715-8,Piers MahonIsmini ChatzitheofilouAndre DekkerX...,2024,https://link.springer.com/article/10.1038/s415...,Article
3,An interpretable deep learning framework for g...,Nature Machine Intelligence,NaN,NaN,NaN,10.1038/s42256-024-00866-y,Shuangxia RenGregory F. CooperLujia ChenXinghu...,2024,https://link.springer.com/article/10.1038/s422...,Article
4,Flexynesis: A deep learning toolkit for bulk m...,Nature Communications,NaN,NaN,NaN,10.1038/s41467-025-63688-5,Bora UyarTaras SavchynAmirhossein Naghsh Nilch...,2025,https://link.springer.com/article/10.1038/s414...,Article


In [ ]:
xls = pd.ExcelFile(file_path)
xls.sheet_names

['TYF1',
 'TYF2',
 'TYF3',
 'PBM1',
 'PBM2',
 'PBM3',
 'PBM4',
 'SCP1',
 'SCP2',
 'SCP3',
 'SCP4',
 'GSCH1',
 'GSCH2',
 'GSCH3',
 'GSCH4',
 'CR2',
 'CR3',
 'CR4',
 'SPG1',
 'SPG2',
 'SPG3',
 'IEE']

In [ ]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 652 entries, 0 to 651
Data columns (total 10 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   Item Title         652 non-null    object 
 1   Publication Title  650 non-null    object 
 2   Book Series Title  0 non-null      float64
 3   Journal Volume     0 non-null      float64
 4   Journal Issue      0 non-null      float64
 5   Item DOI           652 non-null    object 
 6   Authors            580 non-null    object 
 7   Publication Year   652 non-null    int64  
 8   URL                652 non-null    object 
 9   Content Type       652 non-null    object 
dtypes: float64(3), int64(1), object(6)
memory usage: 51.1+ KB


In [ ]:
all_sheets = pd.read_excel(file_path, sheet_name=None)

In [ ]:
dfs = []

for sheet_name, sheet_df in all_sheets.items():
    sheet_df["source_sheet"] = sheet_name
    dfs.append(sheet_df)

raw_df = pd.concat(dfs, ignore_index=True)

In [ ]:
raw_df.shape
raw_df.columns

Index(['Item Title', 'Publication Title', 'Book Series Title',
       'Journal Volume', 'Journal Issue', 'Item DOI', 'Authors',
       'Publication Year', 'URL', 'Content Type', 'source_sheet', 'Cites',
       'Title', 'Year', 'Source', 'Publisher', 'ArticleURL', 'CitesURL',
       'GSRank', 'QueryDate', 'Type', 'DOI', 'ISSN', 'CitationURL', 'Volume',
       'Issue', 'StartPage', 'EndPage', 'ECC', 'CitesPerYear',
       'CitesPerAuthor', 'AuthorCount', 'Age', 'Abstract', 'FullTextURL',
       'RelatedURL', 'Columna 1', 'Columna 2', 'Document Title',
       'Author Affiliations', 'Date Added To Xplore', 'Start Page', 'End Page',
       'ISBNs', 'Funding Information', 'PDF Link', 'Author Keywords',
       'IEEE Terms', 'Mesh_Terms', 'Article Citation Count',
       'Patent Citation Count', 'Reference Count', 'License', 'Online Date',
       'Issue Date', 'Meeting Date', 'Document Identifier'],
      dtype='object')

In [ ]:
def coalesce_columns(df, cols):
    available = [c for c in cols if c in df.columns]
    if not available:
        return None
    result = df[available[0]]
    for c in available[1:]:
        result = result.combine_first(df[c])
    return result

## 2. Normalización de campos bibliográficos

Dado que las distintas bases de datos presentan esquemas de metadatos heterogéneos, se realizó una normalización semántica de encabezados.  
Se definieron columnas canónicas (título, DOI, año, fuente, tipo de documento, resumen, URL y palabras clave), construidas mediante coalescencia de campos equivalentes provenientes de distintas fuentes.

In [ ]:
df = raw_df.copy()

df["title_norm"] = coalesce_columns(df, [
    "Title", "Item Title", "Document Title"
])

df["doi_norm"] = coalesce_columns(df, [
    "DOI", "Item DOI", "Document Identifier"
])

df["year_norm"] = coalesce_columns(df, [
    "Year", "Publication Year", "Online Date", "Issue Date"
])

df["authors_norm"] = coalesce_columns(df, [
    "Authors"
])

df["source_norm"] = coalesce_columns(df, [
    "Source", "Publication Title", "Book Series Title", "Publisher"
])

df["type_norm"] = coalesce_columns(df, [
    "Type", "Content Type"
])

df["abstract_norm"] = coalesce_columns(df, [
    "Abstract"
])

df["url_norm"] = coalesce_columns(df, [
    "ArticleURL", "URL", "FullTextURL", "PDF Link"
])

df["keywords_norm"] = coalesce_columns(df, [
    "Author Keywords", "IEEE Terms", "Mesh_Terms"
])

In [ ]:
df["title_norm"] = (
    df["title_norm"]
    .astype(str)
    .str.lower()
    .str.strip()
)

df["doi_norm"] = (
    df["doi_norm"]
    .astype(str)
    .str.lower()
    .str.strip()
)

df["type_norm"] = (
    df["type_norm"]
    .astype(str)
    .str.lower()
    .str.strip()
)

## 3. Evaluación de completitud de metadatos

Se evaluó la completitud de los campos canónicos.  
Los resultados muestran una cobertura total de títulos y DOIs, una pérdida marginal en el año de publicación y una ausencia esperable de resúmenes en aproximadamente el 77 % de los registros, atribuible a limitaciones propias de ciertas bases de datos técnicas.

In [ ]:
df.isna().mean()[[
    "title_norm",
    "doi_norm",
    "year_norm",
    "abstract_norm"
]]

,0
title_norm,0.000000
doi_norm,0.000000
year_norm,0.001463
abstract_norm,0.769986


## 4. Interpretación técnica de los resultados (diagnóstico)

### 1.1 Completitud de campos canónicos

```
title_norm       0.000000
doi_norm         0.000000
year_norm        0.001463
abstract_norm    0.769986
```

Esto significa:

* **100 % de los registros tienen título** → excelente.
* **100 % tienen DOI** → situación ideal y poco frecuente.
* **>99.8 % tienen año** → pérdida marginal, aceptable.
* **~77 % NO tienen abstract** → esperado (IEEE, GS, Springer, etc.).

👉 **Conclusión**:
El dataset es **óptimo para deduplicación y filtrado estructural**, pero el filtrado semántico por abstract **no puede ser temprano** (correcto dejar LLM al final).

In [ ]:
df[[
    "title_norm",
    "doi_norm",
    "year_norm",
    "source_norm",
    "type_norm",
    "source_sheet"
]].head(10)

,title_norm,doi_norm,year_norm,source_norm,type_norm,source_sheet
0,ai-driven precision oncology: integrating deep...,10.1007/s11760-025-04244-y,2025.0,"Signal, Image and Video Processing",article,TYF1
1,breast cancer histology image repositories in ...,10.1007/978-3-032-10940-8_18,2026.0,Data Science and Applications,conference paper,TYF1
2,a federated learning system for precision onco...,10.1038/s41591-023-02715-8,2024.0,Nature Medicine,article,TYF1
3,an interpretable deep learning framework for g...,10.1038/s42256-024-00866-y,2024.0,Nature Machine Intelligence,article,TYF1
4,flexynesis: a deep learning toolkit for bulk m...,10.1038/s41467-025-63688-5,2025.0,Nature Communications,article,TYF1
5,convergence of evolving artificial intelligenc...,10.1038/s41746-025-01471-y,2025.0,npj Digital Medicine,article,TYF1
6,exploiting artificial intelligence in precisio...,10.1186/s12967-025-07308-2,2025.0,Journal of Translational Medicine,article,TYF1
7,ai-driven multi-omics integration in precision...,10.1007/s10238-025-01965-9,2025.0,Clinical and Experimental Medicine,article,TYF1
8,precision oncology: transforming cancer care t...,10.1007/s12032-025-02817-y,2025.0,Medical Oncology,article,TYF1
9,a systematic analysis of deep learning in geno...,10.1186/s12920-024-01796-9,2024.0,BMC Medical Genomics,article,TYF1


### 1.2 Calidad semántica de los campos

Los ejemplos que muestras confirman que:

* `title_norm` está limpio, en minúsculas, sin ruido.
* `doi_norm` está normalizado correctamente.
* `type_norm` es consistente (`article`, `conference paper`).
* `source_sheet` preserva trazabilidad.

Esto es exactamente lo que se busca en **ETAPA 0 bien ejecutada**.


In [ ]:
# -----------------------------
# ÍNDICE DE COMPLETITUD (ETAPA 0)
# -----------------------------

quality_fields = [
    "title_norm",
    "doi_norm",
    "year_norm",
    "type_norm",
    "source_norm",
    "abstract_norm",
    "keywords_norm"
]

# Indicadores binarios de presencia por campo
for col in quality_fields:
    df[f"{col}_present"] = df[col].notna().astype(int)

# Score de completitud por registro (0–1)
df["metadata_completeness_score"] = (
    df[[f"{c}_present" for c in quality_fields]]
    .mean(axis=1)
)


In [ ]:
df["metadata_completeness_score"].describe()
df.groupby("source_sheet")["metadata_completeness_score"].mean().sort_values()
df["metadata_completeness_score"].value_counts(bins=5).sort_index()

,count
"(0.57, 0.657]",28
"(0.657, 0.743]",7354
"(0.743, 0.829]",0
"(0.829, 0.914]",2146
"(0.914, 1.0]",41


| Intervalo      | Campos presentes | Interpretación            |
| -------------- | ---------------- | ------------------------- |
| (0.427, 0.543] | 3 / 7            | Metadatos muy incompletos |
| (0.543, 0.657] | 4 / 7            | Incompletos               |
| (0.657, 0.771] | 5 / 7            | Aceptables                |
| (0.771, 0.886] | 6 / 7            | Muy buenos                |
| (0.886, 1.0]   | 7 / 7            | Completos                 |

El 97 % de los registros presenta al menos 5 de los 7 metadatos canónicos, evidenciando una alta calidad estructural del corpus. La ausencia de resúmenes se concentra en bases técnicas, lo cual es consistente con la naturaleza de dichas fuentes.

# Etapa N°1

In [ ]:
# Año como numérico
df["year_norm"] = pd.to_numeric(df["year_norm"], errors="coerce")

# Reemplazar strings 'nan' por NaN real
df["doi_norm"] = df["doi_norm"].replace("nan", pd.NA)
df["title_norm"] = df["title_norm"].replace("nan", pd.NA)


In [ ]:
# ----------------------------------
# DEDUPLICACIÓN CON TRAZABILIDAD
# ----------------------------------

df_dedup = df.copy()

# Orden estable por fuente (mismo criterio que antes)
df_dedup = df_dedup.sort_values(by=["source_sheet"])

# Inicialmente todos se conservan
df_dedup["dedup_reason"] = "kept"

# 1. Duplicados por DOI
dup_doi_mask = (
    df_dedup["doi_norm"].notna() &
    df_dedup.duplicated(subset=["doi_norm"], keep="first")
)

df_dedup.loc[dup_doi_mask, "dedup_reason"] = "duplicate_doi"
df_dedup = df_dedup[~dup_doi_mask]

# 2. Duplicados por título
dup_title_mask = (
    df_dedup["title_norm"].notna() &
    df_dedup.duplicated(subset=["title_norm"], keep="first")
)

df_dedup.loc[dup_title_mask, "dedup_reason"] = "duplicate_title"
df_dedup = df_dedup[~dup_title_mask]

df_before = df.shape[0]
df_after = df_dedup.shape[0]

df_before, df_after


(9569, 4114)

## Qué significa el resultado

```
df_before = 9569
df_after  = 4114
```

Esto significa que, tras la deduplicación **con trazabilidad**:

* Partiste de **9.569 registros crudos**
* Conservaste **4.114 registros únicos**
* Eliminaste **5.455 duplicados**

### Método actual (explícito y trazable)

* Elimina duplicados por DOI **primero**.
* Luego elimina duplicados por título **solo entre los supervivientes**.
* Respeta estrictamente la jerarquía:

  * DOI > título

Esto es **más conservador y más correcto metodológicamente**.

---

### Lógica de eliminación de registros con títulos idénticos

La deduplicación bibliográfica se realizó mediante un enfoque **jerárquico y conservador**, con el objetivo de minimizar la pérdida informacional y evitar falsos positivos.

En primer lugar, se aplicó la eliminación de duplicados basada en el identificador persistente **DOI**, considerado una clave de identidad fuerte y unívoca. Todos los registros que compartían el mismo DOI fueron considerados representaciones redundantes de una misma obra, conservándose únicamente la primera ocurrencia según un criterio estable de priorización de fuente.

Una vez eliminados los duplicados por DOI, se procedió a una segunda fase de deduplicación basada en el **título normalizado**. En esta etapa, los registros con títulos idénticos fueron eliminados **únicamente si no poseían un DOI distinto**, es decir, solo se consideraron duplicados aquellos títulos coincidentes entre registros que ya no podían distinguirse mediante un identificador persistente.

Este diseño evita la eliminación de documentos que, aun compartiendo un mismo título, corresponden a obras distintas (por ejemplo, versiones extendidas, artículos derivados de conferencias, erratas publicadas, o publicaciones en distintos canales con DOI propio). De este modo, el título se utiliza como una **clave de identidad débil y secundaria**, subordinada al DOI.

La estrategia adoptada garantiza que la coincidencia de título no prevalezca sobre la existencia de identificadores persistentes distintos, preservando así la integridad del corpus y reduciendo el riesgo de sobre-deduplicación.

In [ ]:
YEAR_MIN = 2018

df_time = df_dedup[df_dedup["year_norm"] >= YEAR_MIN]
df_time.shape


(3808, 75)

In [ ]:
valid_types = [
    "article",
    "journal article",
    "conference paper",
    "conference proceeding",
    "research article"
]


In [ ]:
df_type = df_time[
    df_time["type_norm"]
    .str.contains("|".join(valid_types), na=False)
]

df_type.shape


(2493, 75)

In [ ]:
# Distribución por año
df_type["year_norm"].value_counts().sort_index()

# Distribución por tipo
df_type["type_norm"].value_counts().head(10)

# Cuántos vienen de cada hoja/fuente
df_type["source_sheet"].value_counts()


,count
source_sheet,
SPG1,685
CR2,478
CR3,473
CR4,359
SPG2,158
PBM1,150
PBM4,98
SCP1,30
PBM3,29


In [ ]:
df_stage1 = df_type.copy()
df_stage1.shape

(2493, 75)

In [ ]:
output_path = "/content/drive/MyDrive/Título/bibliografia_etapa1.csv"
df_stage1.to_csv(output_path, index=False)

# Etapa N°2

## Filtrado Temático y Preparación para Validación Semántica

### 1. Filtrado Temático Avanzado

En esta etapa se aplicó un **filtrado temático basado en palabras clave** para refinar los registros bibliográficos previamente depurados y deduplicados (Etapa 1). Se definieron:

* **Palabras clave obligatorias:** artículos que deben contener al menos una de las siguientes: `"precision oncology"`, `"deep learning"`, `"federated learning"`, `"breast cancer"`, `"lung cancer"`.
* **Palabras clave de exclusión:** artículos que contengan `"review"`, `"case report"`, `"editorial"`, `"letter"`, `"conference abstract"`, `"animal model"` fueron eliminados para reducir ruido y enfocarse en artículos primarios de relevancia clínica y técnica.

Se combinó el título normalizado (`title_norm`), el resumen (`abstract_norm`) y las palabras clave (`keywords_norm`) en un solo campo de texto (`text_combined`) y se convirtió todo a minúsculas para garantizar coincidencias consistentes.

El resultado del filtrado temático fue:

* **Antes:** 2.493 registros
* **Después:** 634 registros

Esto asegura que el corpus contenga únicamente artículos **relevantes para precision oncology y aprendizaje automático/federado**, manteniendo trazabilidad con la fuente original (`source_sheet`).

---

### 2. Scoring de Relevancia Temática

Para priorizar los registros dentro del corpus filtrado, se aplicó un **score de relevancia** que combina la presencia de palabras clave complementarias:

* **Técnicas:** machine learning, AI, redes neuronales, radiomics, multi-omics, entre otras.
* **Clínicas/aplicación:** tumor profiling, biomarkers, treatment response, personalized medicine, clinical trial, etc.
* **Metodológicas (federated/distributed):** multi-institutional, privacy-preserving, collaborative learning, secure aggregation, decentralized training.

Se calculó para cada artículo un `relevance_score` como la suma de coincidencias en las tres categorías, además de desglosar en `technical_score`, `clinical_score` y `federated_score` para análisis más detallado.

**Ejemplo de los artículos más relevantes:**

| title_norm                                                  | relevance_score | technical_score | clinical_score | federated_score |
| ----------------------------------------------------------- | --------------- | --------------- | -------------- | --------------- |
| radiomics in soft tissue sarcoma: toward precision oncology | 7               | 4               | 2              | 1               |
| global trends in the use of artificial intelligence         | 6               | 6               | 0              | 0               |
| a foundation model for predicting outcomes of cancer        | 6               | 4               | 2              | 0               |

* **Número total de registros tras scoring:** 634

Este score permite **priorizar los artículos más pertinentes** antes de aplicar cualquier evaluación semántica más avanzada, como la que se realizará localmente mediante un LLM.

---

### 3. Exportación Final

El dataframe final `df_stage2`, que incluye:

* Metadatos originales y normalizados (`title_norm`, `doi_norm`, `year_norm`, `source_norm`, `type_norm`, etc.)
* Campo combinado de texto (`text_combined`)
* Scores de relevancia (`relevance_score`, `technical_score`, `clinical_score`, `federated_score`)

fue exportado a CSV en Google Drive:

```
/content/drive/MyDrive/Título/bibliografia_etapa2.csv
```

* **Número total de registros exportados:** 634
* Este CSV está listo para ser procesado localmente con un LLM para validación semántica adicional y análisis posterior.

In [ ]:
# -----------------------------
# ETAPA 2 - BLOQUE 1: FILTRADO TEMÁTICO AVANZADO (CORREGIDO)
# -----------------------------

import re

# Convertir a string y reemplazar NaN por ""
df_stage1["title_norm"] = df_stage1["title_norm"].astype(str).fillna("")
df_stage1["abstract_norm"] = df_stage1["abstract_norm"].astype(str).fillna("")
df_stage1["keywords_norm"] = df_stage1["keywords_norm"].astype(str).fillna("")

# Crear columna combinada
df_stage1["text_combined"] = (
    df_stage1["title_norm"] + " " +
    df_stage1["abstract_norm"] + " " +
    df_stage1["keywords_norm"]
).str.lower()

# Palabras clave obligatorias
mandatory_keywords = [
    "precision oncology",
    "deep learning",
    "federated learning",
    "breast cancer",
    "lung cancer"
]

# Palabras clave de exclusión
exclude_keywords = [
    "review",
    "case report",
    "editorial",
    "letter",
    "conference abstract",
    "animal model"
]

# Filtrado obligatorio
pattern_mandatory = "|".join([re.escape(k.lower()) for k in mandatory_keywords])
mask_mandatory = df_stage1["text_combined"].str.contains(pattern_mandatory, na=False)

# Excluir registros con palabras clave de exclusión
pattern_exclude = "|".join([re.escape(k.lower()) for k in exclude_keywords])
mask_exclude = ~df_stage1["text_combined"].str.contains(pattern_exclude, na=False)

# Aplicar filtros
df_stage2 = df_stage1[mask_mandatory & mask_exclude].copy()

# Resumen del filtrado
print(f"Registros antes del filtrado temático: {df_stage1.shape[0]}")
print(f"Registros después del filtrado temático: {df_stage2.shape[0]}")

# Primeras filas para inspección
df_stage2[["title_norm", "abstract_norm", "keywords_norm", "source_sheet"]].head(10)


Registros antes del filtrado temático: 2493
Registros después del filtrado temático: 634


,title_norm,abstract_norm,keywords_norm,source_sheet
4899,the future of precision oncology and artificia...,"Purpose: Precision medicine, also known as per...",nan,CR2
4911,ai-driven predictive analytics in precision ag...,The rapid advancement of artificial intelligen...,nan,CR2
4759,towards precision oncology: enhancing cancer s...,Highly complex and multi-dimensional medical d...,nan,CR2
4732,a deep learning framework for causal inference...,Background:: Clinical research is limited by t...,nan,CR2
4828,artificial intelligence and mobile health for ...,"Background: Diabetes is a major health issue, ...",nan,CR2
4843,artificial intelligence in breast oncology,Abstract: Artificial intelligence (AI) is base...,nan,CR2
4975,organ-agnostic precision oncology,nan,nan,CR2
5135,abstract a063: artificial intelligence in cyto...,Cytopathology and histopathology are fields in...,nan,CR2
5153,artificial intelligence in surgery: redefining...,"Previously thought to be science fiction, AI h...",nan,CR2
5162,introducing the <i>jco precision oncology</i> ...,nan,nan,CR2


In [ ]:
# -----------------------------
# ETAPA 2 - BLOQUE 2: SCORE DE RELEVANCIA TEMÁTICA
# -----------------------------

import numpy as np

# Listas de palabras clave complementarias
technical_keywords = [
    "machine learning",
    "artificial intelligence", "ai",
    "neural network", "cnn", "rnn", "transformer",
    "computational oncology",
    "predictive modeling",
    "bioinformatics",
    "radiomics",
    "genomics", "multi-omics",
    "digital pathology"
]

clinical_keywords = [
    "tumor profiling",
    "biomarkers",
    "treatment response",
    "clinical outcome",
    "personalized medicine",
    "therapeutic decision",
    "patient cohort",
    "clinical trial"
]

federated_keywords = [
    "multi-institutional",
    "privacy-preserving",
    "collaborative learning",
    "secure aggregation",
    "decentralized training"
]

# Función para contar coincidencias de palabras clave en el texto
def count_keywords(text, keywords_list):
    count = 0
    for kw in keywords_list:
        if kw.lower() in text:
            count += 1
    return count

# Aplicar la función para cada registro
df_stage2["technical_score"] = df_stage2["text_combined"].apply(lambda x: count_keywords(x, technical_keywords))
df_stage2["clinical_score"] = df_stage2["text_combined"].apply(lambda x: count_keywords(x, clinical_keywords))
df_stage2["federated_score"] = df_stage2["text_combined"].apply(lambda x: count_keywords(x, federated_keywords))

# Relevance score total
df_stage2["relevance_score"] = df_stage2[["technical_score", "clinical_score", "federated_score"]].sum(axis=1)

# Ordenar por relevancia descendente
df_stage2 = df_stage2.sort_values(by="relevance_score", ascending=False)

# Mostrar resumen
print("Resumen de scores de relevancia:")
print(df_stage2[["title_norm", "relevance_score", "technical_score", "clinical_score", "federated_score"]].head(10))
print(f"\nNúmero total de registros tras scoring: {df_stage2.shape[0]}")


Resumen de scores de relevancia:
                                             title_norm  relevance_score  \
2648  radiomics in soft tissue sarcoma: toward preci...                7   
2714  global trends in the use of artificial intelli...                6   
2333  a foundation model for predicting outcomes of ...                6   
2650  machine learning driven multiomics analysis id...                6   
2688  integrated histopathologic modeling of detaile...                6   
2754  a comprehensive benchmarking of machine learni...                6   
4419  precision oncology in the era of genomics and ...                5   
2631  validation of fibroblast activation protein an...                5   
2440  machine learning and computed tomography radio...                5   
2344  ai-driven variant annotation for precision onc...                5   

      technical_score  clinical_score  federated_score  
2648                4               2                1  
2714            

In [ ]:
# -----------------------------
# ETAPA 2 - BLOQUE FINAL: EXPORTACIÓN
# -----------------------------

# Ruta de salida en Google Drive
output_path_stage2 = "/content/drive/MyDrive/Título/bibliografia_etapa2.csv"

# Guardar dataframe final
df_stage2.to_csv(output_path_stage2, index=False)

print(f"Dataset de Etapa 2 exportado correctamente a: {output_path_stage2}")
print(f"Número total de registros exportados: {df_stage2.shape[0]}")

Dataset de Etapa 2 exportado correctamente a: /content/drive/MyDrive/Título/bibliografia_etapa2.csv
Número total de registros exportados: 634


:Finalmente se deja los que tengan un score superior a 4